# 🎤 Speech Enhancement — Step 3: Denoise & Evaluate

This notebook:
1. Mounts Google Drive
2. Loads a trained U-Net model from Drive
3. Denoises one or several noisy WAV files
4. Displays before/after spectrograms and lets you play back the audio
5. **Evaluates** denoising quality using SNR, PESQ, and STOI metrics

### Prerequisites
- Run notebooks 01 and 02 first.
- Your Drive should have `weights/model_unet.h5`
- Place your noisy test WAVs under `data/test/` in Drive.

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
%pip install -q librosa soundfile pesq pystoi

In [ ]:
# ── 3. Clone repo & add src/ to path ──────────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/theweird-kid/speech_enhancement.git'  # ← update
REPO_DIR = '/content/speech_enhancement'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready ✓')

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────
import config as C

# Path to the weights to use for denoising
WEIGHTS = os.path.join(C.WEIGHTS_DIR, 'model_unet.h5')
print(f'Weights path: {WEIGHTS}')
print(f'Test input dir: {C.AUDIO_INPUT_DIR}')
print(f'Predictions dir: {C.PRED_DIR}')

os.makedirs(C.PRED_DIR, exist_ok=True)

In [ ]:
# ── 5. Upload a noisy WAV directly (optional alternative to Drive) ─────────
# from google.colab import files
# uploaded = files.upload()
# input_file = list(uploaded.keys())[0]   # path of uploaded file
# print('Uploaded:', input_file)

In [ ]:
# ── 6. Denoise a single file ───────────────────────────────────────────────
from predict import denoise_audio

# Change filename to your test file:
input_wav  = os.path.join(C.SOUND_DIR, 'sp16_babble_sn10.wav')   # ← change this
output_wav = os.path.join(C.PRED_DIR,  'denoised_sp16_babble_sn10.wav')

audio_out = denoise_audio(
    input_path   = input_wav,
    output_path  = output_wav,
    weights_path = WEIGHTS,
)
print('Denoising complete!')

In [ ]:
# ── 7. Play back & compare ────────────────────────────────────────────────
import IPython.display as ipd
import soundfile as sf

def play(path, label):
    data, sr = sf.read(path)
    print(f'\n🔊  {label}')
    display(ipd.Audio(data, rate=sr))

play(input_wav,  'Input  (noisy)')
play(output_wav, 'Output (denoised)')

In [ ]:
# ── 8. Visualise spectrogram before vs after ───────────────────────────────
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

def load_spec(wav_path, sr=C.SAMPLE_RATE):
    y, _ = librosa.load(wav_path, sr=sr)
    S    = librosa.stft(y, n_fft=C.N_FFT, hop_length=C.HOP_LENGTH_FFT)
    return librosa.amplitude_to_db(np.abs(S), ref=np.max)

S_noisy   = load_spec(input_wav)
S_denoised = load_spec(output_wav)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
librosa.display.specshow(S_noisy,   sr=C.SAMPLE_RATE,
                         hop_length=C.HOP_LENGTH_FFT,
                         x_axis='time', y_axis='hz', ax=axes[0], cmap='magma')
axes[0].set_title('Noisy Voice Spectrogram')

librosa.display.specshow(S_denoised, sr=C.SAMPLE_RATE,
                         hop_length=C.HOP_LENGTH_FFT,
                         x_axis='time', y_axis='hz', ax=axes[1], cmap='magma')
axes[1].set_title('Denoised Voice Spectrogram')

plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Batch denoise all WAVs in test folder ───────────────────────────────
test_files = [f for f in os.listdir(C.AUDIO_INPUT_DIR) if f.endswith('.wav')]
print(f'Found {len(test_files)} test files')

for fname in test_files:
    in_path  = os.path.join(C.AUDIO_INPUT_DIR, fname)
    out_name = 'denoised_' + fname
    out_path = os.path.join(C.PRED_DIR, out_name)
    print(f'Processing {fname} …')
    denoise_audio(in_path, out_path, weights_path=WEIGHTS)
    print(f'  → {out_path}')

print('\nAll files processed!')

---
## 📊 Performance Evaluation

Denoise the QC audio pair from `create_data()` and measure quality using:

| Metric | What it measures | Range |
|--------|-----------------|-------|
| **SNR** | Signal-to-noise ratio improvement | Higher = better (dB) |
| **PESQ** | Perceptual speech quality (ITU standard) | –0.5 → 4.5, higher = better |
| **STOI** | Speech intelligibility | 0 → 1, higher = better |

In [ ]:
# ── 10. Denoise QC audio for evaluation ───────────────────────────────────
import pandas as pd
from pesq import pesq as pesq_fn
from pystoi import stoi as stoi_fn

CLEAN_WAV    = os.path.join(C.SOUND_DIR, 'clean_voice_long.wav')
NOISY_WAV    = os.path.join(C.SOUND_DIR, 'noisy_voice_long.wav')
DENOISED_WAV = os.path.join(C.PRED_DIR,  'denoised_noisy_voice_long.wav')

print('Denoising QC noisy audio for evaluation …')
_ = denoise_audio(
    input_path   = NOISY_WAV,
    output_path  = DENOISED_WAV,
    weights_path = WEIGHTS,
)
print('QC denoising complete ✓')

In [ ]:
# ── 11. Compute SNR / PESQ / STOI ─────────────────────────────────────────
SR = C.SAMPLE_RATE

def load_wav(path, sr=SR):
    y, _ = librosa.load(path, sr=sr)
    return y.astype(np.float32)

def align(a, b):
    n = min(len(a), len(b))
    return a[:n], b[:n]

def compute_snr(clean, test):
    noise = clean - test
    return 10 * np.log10(np.sum(clean**2) / (np.sum(noise**2) + 1e-8))

def compute_metrics(clean, test, sr=SR):
    c, t = align(clean, test)
    return {
        'SNR (dB)': round(compute_snr(c, t), 3),
        'PESQ'    : round(pesq_fn(sr, c, t, 'nb'), 3),
        'STOI'    : round(stoi_fn(c, t, sr, extended=False), 3),
    }

clean_full    = load_wav(CLEAN_WAV)
noisy_full    = load_wav(NOISY_WAV)
denoised_full = load_wav(DENOISED_WAV)

# PESQ / STOI are slow on very long audio — evaluate on first 30 s
MAX_SEC = 30
MAX_N   = MAX_SEC * SR

clean_a    = clean_full[:MAX_N]
noisy_a    = noisy_full[:MAX_N]
denoised_a = denoised_full[:MAX_N]

print(f'Full audio : {len(clean_full)/SR:.1f} s')
print(f'Evaluating on first {MAX_SEC}s ({MAX_N} samples)\n')

print('=== Noisy vs Clean (before enhancement) ===')
before = compute_metrics(clean_a, noisy_a)
for k, v in before.items():
    print(f'  {k}: {v}')

print('\n=== Denoised vs Clean (after enhancement) ===')
after = compute_metrics(clean_a, denoised_a)
for k, v in after.items():
    print(f'  {k}: {v}')

# Summary table
metrics = ['SNR (dB)', 'PESQ', 'STOI']
df = pd.DataFrame({
    'Metric':      metrics,
    'Before':      [before[m] for m in metrics],
    'After':       [after[m]  for m in metrics],
})
df['Improvement'] = (df['After'] - df['Before']).round(3)
print('\n', df.to_string(index=False))

In [ ]:
# ── 12. Bar chart: before vs after ─────────────────────────────────────────
before_vals = [before[m] for m in metrics]
after_vals  = [after[m]  for m in metrics]

x, width = np.arange(len(metrics)), 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, before_vals, width, label='Noisy (before)',   color='#e05c5c')
bars2 = ax.bar(x + width/2, after_vals,  width, label='Denoised (after)', color='#4caf7d')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Speech Enhancement — Before vs After', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.savefig(os.path.join(C.WEIGHTS_DIR, 'performance_chart.png'), dpi=120)
plt.show()

print(df.to_string(index=False))